# Step 1.3: SpeechArenaBench Hindi (test only) and BVCC inspection

**Run this on the server** (kernel = the `hinglishtts` venv; it needs `ipykernel`).

| Part | Cells | What it does | Writes anything? |
|---|---|---|---|
| A1-A3 | layout | Lists SpeechArenaBench files and sizes; shows the Hindi shard schema and 3 metadata rows (no audio) | No |
| A4 | dry run | For the first `SHARD_LIMIT` Hindi shards, reads only the small text columns and shows which pairs are kept under our tier rule and how many row groups would be downloaded | No |
| A5 | download | Downloads only kept pairs into `datasets/speecharena_hi_AB/`. **Off by default** (`RUN_DOWNLOAD = False`) | Only when you switch it on |
| B | BVCC | Lists the archive and prints its fetch scripts as text. Nothing is extracted. `01_get.py` is **read, never run** | No |
| C | summary | Prints one summary block for the ledger | No |

**Rule applied:** SpeechArenaBench is **test-only**. We keep a pair only if **both** systems are in tier A or B (Indic F5, Gemini 2.5 Pro TTS, GPT 4o Mini TTS, Sonic 3). Pairs involving Bulbul, Eleven Labs or Speech 2.8 HD (tier C, vendor terms) are dropped and never written to disk.
Expected result for all 70 Hindi shards: **4,444 pairs, of which 1,159 are mixed Hindi-English** (from the counting script).

In [ ]:
import collections
import hashlib
import json
import shutil
import sys
import tarfile
import time
from pathlib import Path

import huggingface_hub
import pyarrow
import pyarrow.parquet
from huggingface_hub import HfApi
from huggingface_hub import HfFileSystem

ROOT = Path.home() / "projects" / "hinglish-cs-tts"
DS = ROOT / "datasets"
SAB_REPO = "ai4bharat/SpeechArenaBench"
LANG = "hi"
OUT = DS / "speecharena_hi_AB"
BVCC_MAIN = DS / "bvcc" / "main.tar.gz"

# everything shown with log() is also saved to output.log, so no result lives only in kernel memory
OUTDIR = ROOT / "analysis" / "step1_3"
OUTDIR.mkdir(parents=True, exist_ok=True)


def log(*parts):
    """Print a line and append it to output.log."""
    line = " ".join([str(part) for part in parts])
    print(line)
    with open(OUTDIR / "output.log", "a") as log_file:
        log_file.write(line + "\n")


log("python", sys.version.split()[0], "| huggingface_hub", huggingface_hub.__version__, "| pyarrow", pyarrow.__version__)
log("Hugging Face user:", HfApi().whoami()["name"])
log("BVCC archive present:", BVCC_MAIN.exists())
log("free disk (TB):", round(shutil.disk_usage(ROOT).free / 1e12, 2))
log("outputs and results are saved under", OUTDIR)

## Part A: SpeechArenaBench layout (read-only)

In [ ]:
# A1. Files and sizes per language folder (file list only, no data downloaded)
api = HfApi()
info = api.dataset_info(SAB_REPO, files_metadata=True)
SAB_SHA = info.sha   # dataset version, goes into the ledger
log("dataset revision:", SAB_SHA, "| last modified:", info.last_modified)

files_per_folder = {}
bytes_per_folder = {}
for sibling in info.siblings:
    if "/" in sibling.rfilename:
        folder = sibling.rfilename.split("/")[0]
    else:
        folder = "(root)"
    files_per_folder[folder] = files_per_folder.get(folder, 0) + 1
    bytes_per_folder[folder] = bytes_per_folder.get(folder, 0) + (sibling.size or 0)
for folder in sorted(files_per_folder):
    log(f"{folder:10} {files_per_folder[folder]:4} files {bytes_per_folder[folder] / 1e9:8.1f} GB")

hindi_by_name = {}
for sibling in info.siblings:
    if sibling.rfilename.startswith(f"{LANG}/") and sibling.rfilename.endswith(".parquet"):
        hindi_by_name[sibling.rfilename] = sibling
hi_names = sorted(hindi_by_name)
hi_files = [hindi_by_name[name] for name in hi_names]
hi_bytes = sum([sibling.size for sibling in hi_files])
log(f"\nHindi: {len(hi_files)} shards, {hi_bytes / 1e9:.1f} GB total")
log("first shards:", hi_names[:3])

In [ ]:
# A2. Schema of the first Hindi shard (reads only the parquet footer)
fs = HfFileSystem()
shard_paths = [f"datasets/{SAB_REPO}/{name}" for name in hi_names]
with fs.open(shard_paths[0], "rb") as f:
    parquet_file = pyarrow.parquet.ParquetFile(f)
    log(parquet_file.schema_arrow)
    log("rows:", parquet_file.metadata.num_rows, "| row groups:", parquet_file.num_row_groups)
    first_sizes = []
    for i in range(min(5, parquet_file.num_row_groups)):
        first_sizes.append(parquet_file.metadata.row_group(i).num_rows)
    log("rows per row group (first 5):", first_sizes)

In [ ]:
# A3. Three metadata rows WITHOUT audio. Audio columns hold raw bytes or {bytes, path} structs.
def is_audio_type(column_type):
    """True for the column types that hold audio."""
    if pyarrow.types.is_binary(column_type) or pyarrow.types.is_large_binary(column_type):
        return True
    if pyarrow.types.is_struct(column_type):
        field_names = set()
        for i in range(column_type.num_fields):
            field_names.add(column_type.field(i).name)
        return field_names == {"bytes", "path"}
    return False


META_COLS = []
AUDIO_COLS = []
with fs.open(shard_paths[0], "rb") as f:
    parquet_file = pyarrow.parquet.ParquetFile(f)
    for field in parquet_file.schema_arrow:
        if is_audio_type(field.type):
            AUDIO_COLS.append(field.name)
        else:
            META_COLS.append(field.name)
    sample_rows = parquet_file.read_row_group(0, columns=META_COLS).to_pylist()[:3]
log("metadata columns:", META_COLS)
log("audio columns   :", AUDIO_COLS)
for row in sample_rows:
    log(json.dumps(row, ensure_ascii=False, default=str)[:800])
    log()

## A4. Tier rule and dry run (reads only small text columns)

The dry run answers: how many pairs survive the rule, how many are mixed Hindi-English, and how many row groups hold at least one kept pair (those are the only ones the download step would fetch).
Start with `SHARD_LIMIT = 3`. Set it to `None` to plan all 70 shards (the counting script needed about 12 minutes for text columns).

In [ ]:
TIER_A = {"Indic F5"}
TIER_B = {"Gemini 2.5 Pro TTS", "GPT 4o Mini TTS", "Sonic 3"}
TIER_C = {"Bulbul V3 Beta", "Eleven Labs v3", "Speech 2.8 HD"}   # excluded: vendor terms
ALLOWED = TIER_A | TIER_B


def has_devanagari(text):
    """True if the text has a Devanagari character (Unicode block U+0900 to U+097F)."""
    for ch in text:
        if "ऀ" <= ch <= "ॿ":
            return True
    return False


def has_latin(text):
    """True if the text has an English letter A-Z or a-z."""
    for ch in text:
        if ch.isascii() and ch.isalpha():
            return True
    return False


def kind(sentence):
    """mixed (Devanagari and Latin letters), latin_only, dev_only or other."""
    devanagari = has_devanagari(sentence)
    latin = has_latin(sentence)
    if devanagari and latin:
        return "mixed"
    if latin:
        return "latin_only"
    if devanagari:
        return "dev_only"
    return "other"


SHARD_LIMIT = 3          # None = all shards


def plan_shard(path):
    """Read only the text columns of one shard and decide which rows are kept."""
    with fs.open(path, "rb") as f:
        parquet_file = pyarrow.parquet.ParquetFile(f)
        rows = parquet_file.read(columns=["model_a", "model_b", "sentence"]).to_pylist()
        rg_rows = []
        for i in range(parquet_file.num_row_groups):
            rg_rows.append(parquet_file.metadata.row_group(i).num_rows)
    keep = []
    pairs = collections.Counter()
    mixed_kept = 0
    unknown_systems = set()
    for row in rows:
        kept = row["model_a"] in ALLOWED and row["model_b"] in ALLOWED
        keep.append(kept)
        if kept:
            pairs["|".join(sorted([row["model_a"], row["model_b"]]))] += 1
            if kind(row["sentence"] or "") == "mixed":
                mixed_kept += 1
        for system in [row["model_a"], row["model_b"]]:
            if system not in ALLOWED and system not in TIER_C:
                unknown_systems.add(system)
    return {
        "path": path, "rows": len(keep), "keep": keep, "rg_rows": rg_rows, "pairs": pairs,
        "mixed_kept": mixed_kept, "unknown_systems": unknown_systems,
    }


def row_groups_to_read(plan):
    """How many row groups hold at least one kept row."""
    count = 0
    offset = 0
    for n in plan["rg_rows"]:
        if any(plan["keep"][offset:offset + n]):
            count += 1
        offset += n
    return count


if SHARD_LIMIT is None:
    targets = shard_paths
else:
    targets = shard_paths[:SHARD_LIMIT]
PLAN_CACHE = DS / "speecharena_hi_plan_cache.json"


def load_plan_cache():
    """The plans saved by the download cell, or None if they are for another dataset revision or shard limit."""
    saved = json.loads(PLAN_CACHE.read_text())
    if saved["revision"] != SAB_SHA or saved["shard_limit"] != SHARD_LIMIT:
        return None
    plans = []
    for plan in saved["plans"]:
        plan["pairs"] = collections.Counter(plan["pairs"])
        plan["unknown_systems"] = set(plan["unknown_systems"])
        plans.append(plan)
    return plans


plans = None
if PLAN_CACHE.exists():
    plans = load_plan_cache()
t0 = time.time()
if plans is None:
    plans = []
    for path in targets:
        plans.append(plan_shard(path))
        print(f"planned {len(plans)}/{len(targets)}", end="\r")
else:
    log("plans loaded from the cache file (no re-planning needed)")

log(f"{'shard':34} {'rows':>6} {'kept':>6} {'mixed':>6}  row groups to read / total")
total_rows = 0
total_kept = 0
total_mixed = 0
unknown = set()
for plan in plans:
    kept = sum(plan["keep"])
    name = plan["path"].split("/")[-1]
    log(f"{name:34} {plan['rows']:6d} {kept:6d} {plan['mixed_kept']:6d}  {row_groups_to_read(plan)} / {len(plan['rg_rows'])}")
    total_rows += plan["rows"]
    total_kept += kept
    total_mixed += plan["mixed_kept"]
    unknown.update(plan["unknown_systems"])
log(f"\n{len(plans)} shard(s) in {time.time() - t0:.0f}s: {total_kept} of {total_rows} pairs kept "
    f"({100 * total_kept / max(1, total_rows):.0f}%), {total_mixed} mixed")
log("system names not in any tier (should be empty):", unknown)
if SHARD_LIMIT is None:
    if total_kept == 4444 and total_mixed == 1159:
        log("expected for all shards: 4444 pairs kept, 1159 mixed  -> MATCH")
    else:
        log("expected for all shards: 4444 pairs kept, 1159 mixed  -> DIFFERENT, investigate")

## A5. Download only the kept pairs (OFF by default)

Do a trial first: leave `SHARD_LIMIT = 3` above, re-run the dry-run cell, then set `RUN_DOWNLOAD = True` and run this cell. Check the output looks right before planning all 70 shards.
For each shard it reads only the row groups that contain at least one kept pair, keeps only the kept rows, adds `src_shard` and `src_row` columns for provenance, and writes to `datasets/speecharena_hi_AB/`. Tier C audio is read in memory only when it shares a row group with kept rows, and is **never written to disk**.

In [ ]:
RUN_DOWNLOAD = False     # <- set True to download the kept pairs
HASH_OUTPUT = True       # sha256 of each written file, saved in the manifest
BLOCK_MB = 32            # read each shard in big 32 MB blocks instead of many small requests
PLAN_CACHE = DS / "speecharena_hi_plan_cache.json"   # same path as in the planning cell, so this cell also works on its own


def save_plan_cache(plans):
    """Save the plans, so a restart can reload them instead of repeating the ~20 minute planning."""
    saved_plans = []
    for plan in plans:
        saved = dict(plan)
        saved["pairs"] = dict(plan["pairs"])
        saved["unknown_systems"] = sorted(plan["unknown_systems"])
        saved_plans.append(saved)
    PLAN_CACHE.write_text(json.dumps({"revision": SAB_SHA, "shard_limit": SHARD_LIMIT, "plans": saved_plans}))


def log_run(message):
    """log() a message and also append it to run.log next to the downloaded files."""
    log(message)
    with open(OUT / "run.log", "a") as run_log:
        run_log.write(message + "\n")


def sha256_of(path):
    """sha256 of a file, read in 4 MB pieces."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            piece = f.read(4 * 1024 * 1024)
            if not piece:
                break
            h.update(piece)
    return h.hexdigest()


def process_shard(plan):
    """Download the kept rows of one shard. Returns (status, number of pairs)."""
    name = plan["path"].split("/")[-1]
    final = OUT / name
    tmp = OUT / (name + ".tmp")
    expected = sum(plan["keep"])
    if expected == 0:
        return "no kept pairs", 0
    if final.exists() and pyarrow.parquet.ParquetFile(final).metadata.num_rows == expected:
        # finished in an earlier run (a shard gets its final name only when it is complete)
        return "already done", expected
    tmp.unlink(missing_ok=True)
    writer = None
    offset = 0
    written = 0
    with fs.open(plan["path"], "rb", block_size=BLOCK_MB * 1024 * 1024) as f:
        parquet_file = pyarrow.parquet.ParquetFile(f)
        for group, n in enumerate(plan["rg_rows"]):
            mask = plan["keep"][offset:offset + n]
            if any(mask):
                # only rows of allowed systems are kept
                table = parquet_file.read_row_group(group).filter(pyarrow.array(mask))
                row_numbers = []
                for j in range(n):
                    if mask[j]:
                        row_numbers.append(offset + j)
                table = table.append_column("src_shard", pyarrow.array([name] * len(row_numbers)))
                table = table.append_column("src_row", pyarrow.array(row_numbers, type=pyarrow.int64()))
                if writer is None:
                    writer = pyarrow.parquet.ParquetWriter(tmp, table.schema)
                writer.write_table(table)
                written += len(row_numbers)
            offset += n
    writer.close()
    tmp.replace(final)   # a half-written shard never carries the final name
    return "wrote", written


if not RUN_DOWNLOAD:
    log("RUN_DOWNLOAD is False: nothing downloaded or written.")
else:
    OUT.mkdir(parents=True, exist_ok=True)
    save_plan_cache(plans)
    log_run(f"start: {len(plans)} shards, {BLOCK_MB} MB blocks, {time.strftime('%Y-%m-%d %H:%M:%S')}")
    t_start = time.time()
    failed = []
    done = 0
    for plan in plans:
        name = plan["path"].split("/")[-1]
        try:
            status, count = process_shard(plan)
            done += 1
            log_run(f"[{done}/{len(plans)}] {name}: {status} ({count} pairs) | elapsed {time.time() - t_start:.0f}s")
        except Exception as error:
            # one failed shard does not stop the others
            failed.append(name)
            log_run(f"FAILED {name}: {type(error).__name__}: {str(error)[:200]}")

    if failed:
        log_run(f"{len(failed)} shard(s) failed: {sorted(failed)}. Run this cell again: finished shards are skipped, only these are retried.")
    else:
        files = {}
        kept_total = 0
        mixed_total = 0
        pair_counts = collections.Counter()
        for plan in plans:
            expected = sum(plan["keep"])
            kept_total += expected
            mixed_total += plan["mixed_kept"]
            pair_counts.update(plan["pairs"])
            if expected:
                name = plan["path"].split("/")[-1]
                rows = pyarrow.parquet.ParquetFile(OUT / name).metadata.num_rows
                files[name] = {"rows": rows, "bytes": (OUT / name).stat().st_size}
        if HASH_OUTPUT:
            for name in files:
                files[name]["sha256"] = sha256_of(OUT / name)
        manifest = {
            "source": f"https://huggingface.co/datasets/{SAB_REPO}", "revision": SAB_SHA, "lang": LANG,
            "rule": "keep a pair only if both systems are in tier A or B; tier C never written",
            "tier_a": sorted(TIER_A), "tier_b": sorted(TIER_B), "tier_c_excluded": sorted(TIER_C),
            "shards_processed": len(plans), "all_shards": SHARD_LIMIT is None,
            "pairs_kept": kept_total, "mixed_kept": mixed_total, "pairs_by_system_pair": dict(pair_counts),
            "files": files, "workers": 1, "finished": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        }
        (OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
        log_run(f"\nDone: {kept_total} pairs kept ({mixed_total} mixed) in {time.time() - t_start:.0f}s. Manifest: {OUT / 'manifest.json'}")
        if kept_total == 4444 and mixed_total == 1159:
            log_run("expected for all shards: 4444 pairs, 1159 mixed -> MATCH")
        else:
            log_run("expected for all shards: 4444 pairs, 1159 mixed -> DIFFERENT, investigate")

## Part B: BVCC (read-only)

The VoiceMOS 2022 archive contains scripts that fetch the Blizzard audio. This part **only reads** them (as text, straight from the archive; nothing is extracted to disk and nothing is executed).
Look for: what URLs they download from, how much, whether they ask for a licence or registration, and whether the wav files are already inside `main.tar.gz`.

In [ ]:
# B1. Archive layout, ignoring macOS "._" metadata files
tf = tarfile.open(BVCC_MAIN)
members = []
for member in tf.getmembers():
    if "/._" not in member.name and not member.name.startswith("._"):
        members.append(member)
names = [member.name for member in members]
wav_count = 0
for name in names:
    if name.endswith(".wav"):
        wav_count += 1
log("entries:", len(names), "| wav files:", wav_count)

log("\nby top-level folder/file:")
top = collections.Counter()
extensions = collections.Counter()
for name in names:
    top["/".join(name.split("/")[:2])] += 1
    extensions[Path(name).suffix] += 1
for key, count in top.most_common(25):
    log(f"{count:6d}  {key}")
log("\nextensions:", extensions.most_common(8))

In [ ]:
# B2. Print the fetch scripts and notes as text. NOTHING IS RUN.
def show(suffix, max_chars=7000):
    """Print the first archive member whose name ends with suffix, as text."""
    found = None
    for member in members:
        if member.name.endswith(suffix):
            found = member
            break
    if found is None:
        log(f"--- {suffix}: not found\n")
        return
    text = tf.extractfile(found).read().decode("utf-8", errors="replace")
    log(f"######## {found.name}  ({found.size} bytes)")
    if len(text) > max_chars:
        log(text[:max_chars] + "\n... [truncated]")
    else:
        log(text)
    log()


for suffix in ["01_get.py", "get_transcripts.py", "INFO_ESPNETTTS.txt", "install_sv56.sh"]:
    show(suffix)
for member in members:
    name_upper = member.name.upper()
    is_notes = "README" in name_upper or "LICENSE" in name_upper or "INFO" in name_upper
    if is_notes and member.isfile():
        log("other notes file:", member.name, member.size, "bytes")

## Part C: summary for the ledger

In [ ]:
plan_rows = 0
plan_kept = 0
plan_mixed = 0
for plan in plans:
    plan_rows += plan["rows"]
    plan_kept += sum(plan["keep"])
    plan_mixed += plan["mixed_kept"]
log("=== SUMMARY ===")
log("SAB revision:", SAB_SHA)
log("SAB Hindi shards:", len(hi_files), "| Hindi GB:", round(hi_bytes / 1e9, 1))
log("SAB metadata columns:", META_COLS)
log("SAB audio columns:", AUDIO_COLS)
log("dry run: shards", len(plans), "| kept", plan_kept, "of", plan_rows, "| mixed kept", plan_mixed, "| shard limit", SHARD_LIMIT)
log("download ran:", RUN_DOWNLOAD, "| output dir exists:", OUT.exists())
log("BVCC entries:", len(names), "| wav files in main.tar.gz:", wav_count)
log("=== END ===")